# Dynamic programming (DP)

Idea of breaking down a larger problems recursively into subproblems

Programming: optimising a “program”, i.e. a policy

Breaking into subproblems:
- Solve the subproblems
- Combine solutions to subproblems

Dynamic Programming is a very general solution method for problems which have two properties:
- Optimal substructure
    Optimal solution can be decomposed into subproblems
- Overlapping subproblems
    Subproblems recur many times
    Solutions can be cached and reused

Markov decision processes satisfy both properties
- Bellman equation gives recursive decomposition
- Value function stores and reuses solutions


## Planning by DP

Dynamic programming assumes full knowledge of the MDP (Requires a perfect model of env.)
It is used for planning in an MDP. 

Planning = simulate future trajectories using a model of env to improve policy/value estimate. Instead of learning purely from real experience, the agent "thinks ahead" by querying an internal model

Full knowledge: 
- transitions: next state
- rewards



Nb: DP used to solve other problems like sequence alignement (string), graph algo (shortest path), graphical model (viterbi),...


## Iterative Policy evaluation (Prediction)

Problem: evaluate a given policy $\pi$
Solution: iterative application of Bellman expectation

v1 -> v2 -> v3 -> ... -> $v_\pi$


Use synchronous backup:
- at each iteration k
- for all states (synchronous)
- update $v_{k+1}(s)$ from $v_k(s')$
- s' is successor state




How to compute action-value function

From previous section we know:

<img src="../image-66.png" width="500px" /><div/>


State-value function is the average of all the available action-values, weighted by how likely we are to choose them under our current policy:

We apply the Bellmann equation as an update rule, and compute for each state

<img src="../image-67.png" width="500px" /><div/>



### Example with simple grid

<img src="../image-76.png" width="500px" /><div/>

- Undiscounted episodic MDP (γ = 1)
- Nonterminal states 1,...,14
- One terminal state (shown twice as shaded squares)
- Actions leading out of the grid leave state unchanged
- Reward is −1 until the terminal state is reached
- Agent follows uniform random policy (1/4 (actions)=0.25)
    


k=0: Value initialized to 0
k=1: value = reward (synchronous -> we evaluate all states)
k=2: 
    -2 (immediate reward + prev value)
    -   

<img src="../image-77.png" width="500px" /><div/>

<img src="../image-78.png" width="500px" /><div/>



<img src="../image-68.png" width="500px" /><div/>

###### Code implementation

Bellman equation to update V: 

$V(s)\leftarrow r(s)+\gamma\sum_{s'}P(s'|s)V(s')$

In [7]:
import numpy as np

# ---------- MDP definition (dicts) ----------
states = ["Class1", "Class2", "Class3", "Facebook", "Pub", "Pass", "Sleep"]
terminal_states = ["Sleep"]  # , "Pass"]
state_names = ["C1", "C2", "C3", "FB", "Sleep"]  
    
S = list(state_names)  # ["C1", "C2", "C3", "FB", "Sleep"]
terminal_states_mdp = {"Sleep"}

A = {
    "C1": ["study", "facebook"],
    "C2": ["study", "sleep"],
    "C3": ["study", "pub"],
    "FB": ["quit", "facebook"],
    "Sleep": [],
}

# Transition model: P[(s,a)] = [(prob, s_next), ...]
P_sa = {
    ("C1", "study"): [(1.0, "C2")],
    ("C1", "facebook"): [(1.0, "FB")],
    ("C2", "study"): [(1.0, "C3")],
    ("C2", "sleep"): [(1.0, "Sleep")],
    # Study in C3 gives +10 then day ends
    ("C3", "study"): [(1.0, "Sleep")],
    # Pub in C3 transitions back to class states
    ("C3", "pub"): [(0.2, "C1"), (0.4, "C2"), (0.4, "C3")],
    ("FB", "quit"): [(1.0, "C1")],
    ("FB", "facebook"): [(1.0, "FB")],
}

# Reward model: R[(s,a)] = immediate reward for taking action a in s
R_sa = {
    ("C1", "study"): -2.0,
    ("C1", "facebook"): -1.0,
    ("C2", "study"): -2.0,
    ("C2", "sleep"): 0.0,
    ("C3", "study"): 10.0,
    ("C3", "pub"): 1.0,
    ("FB", "quit"): 0.0,
    ("FB", "facebook"): -1.0,
}

r = {
    "Class1": -2.0,
    "Class2": -2.0,
    "Class3": -2.0,
    "Facebook": -1.0,
    "Pub": 1.0,
    "Sleep": 0.0,
    "Pass": 10.0,
}

def actions(s):
    return A[s]


def transitions_sa(s, a):
    return P_sa[(s, a)]


def reward_sa(s, a):
    return float(R_sa[(s, a)])


# ---------- Bellman operators ----------


def bellman_expectation_backup_v(V, pi, gamma):
    """Bellman expectation equation for V under policy pi.

    V(s) = Σ_a π(a|s) Σ_{s'} P(s'|s,a) [ r(s,a) + γ V(s') ]
    """
    V_new = {}
    for s in S:
        if s in terminal_states_mdp:
            V_new[s] = 0.0
            continue
        v = 0.0
        for a, p_a in pi[s].items():
            q_sa = 0.0
            for p, s_next in transitions_sa(s, a):
                q_sa += p * (reward_sa(s, a) + gamma * V[s_next])
            v += p_a * q_sa
        V_new[s] = v
    return V_new


def q_from_v(V, gamma):
    """Compute Q(s,a) from V: Q(s,a) = Σ_{s'} P(s'|s,a) [ r(s,a) + γ V(s') ]."""
    Q = {s: {} for s in S}
    for s in S:
        if s in terminal_states_mdp:
            continue
        for a in actions(s):
            q = 0.0
            for p, s_next in transitions_sa(s, a):
                q += p * (reward_sa(s, a) + gamma * V[s_next])
            Q[s][a] = q
    return Q


# ---------- Dynamic programming algorithms ----------


def normalize_policy(pi):
    """Ensure pi[s] is a valid distribution over actions(s)."""
    out = {}
    for s in S:
        if s in terminal_states_mdp:
            out[s] = {}
            continue
        if s not in pi or not pi[s]:
            # default uniform
            acts = actions(s)
            out[s] = {a: 1.0 / len(acts) for a in acts}
            continue

        # keep only valid actions
        probs = {a: float(p) for a, p in pi[s].items() if a in set(actions(s))}
        total = sum(probs.values())
        if total <= 0:
            acts = actions(s)
            out[s] = {a: 1.0 / len(acts) for a in acts}
        else:
            out[s] = {a: p / total for a, p in probs.items()}
    return out

def policy_evaluation(pi, gamma=1.0, theta=1e-12, max_iters=100_000):
    """Iterative policy evaluation using the Bellman expectation backup."""
    pi = normalize_policy(pi)
    V = {s: 0.0 for s in S}

    for _ in range(max_iters):
        V_new = bellman_expectation_backup_v(V, pi, gamma)
        delta = max(abs(V_new[s] - V[s]) for s in S)
        V = V_new
        if delta < theta:
            break

    Q = q_from_v(V, gamma)
    return V, Q

transitions_with_terminal_states = {s: [] for s in states}


# Uniform random policy (matches the earlier MDP->MRP conversion)
pi_uniform = {s: {a: 1.0 / len(actions(s)) for a in actions(s)} for s in S}

V_pi, Q_pi = policy_evaluation(pi_uniform, gamma=1.0, theta=1e-12)
print("V^pi (uniform random):")
for s in S:
    print(f"  {s:>5}: {V_pi[s]: .6f}")

V^pi (uniform random):
     C1: -1.307692
     C2:  2.692308
     C3:  7.384615
     FB: -2.307692
  Sleep:  0.000000


#### Policy improvement

From predicting value (estimation) -> move to control problem



Similarly, the action-value function is the average value of the states the agent might transition to given the chosen action, but weighted instead by their transition probabilities

<img src="../image-69.png" width="500px" /><div/>

<img src="../image-44.png" width="600px" /><div/>

We do a one step policy improvement

#### Backup diagram

<img src="../image-75.png" width="600px" /><div/>




So action-value is the immediate reward for taking that action, plus the expected value of the next state, over all possible successor states

If this is better than the value of π, we are better off changing our policy in such a way. And we can indeed answer such question via the policy improvement theorem, which states that if, for all states s:

<img src="../image-70.png" width="200px" /><div/>

The improved policy is better:

<img src="../image-71.png" width="200px" /><div/>




### Policy iteration

chain these improvement steps with policy evaluation


<img src="../image-72.png" width="700px" /><div/>

Each of the resulting policies are monotonically improving, and — since finite MDPs only have finite states — this process must converge to the optimal policy eventually!

So first algorithm for solving an RL problem — called policy iteration


<img src="../image-73.png" width="700px" /><div/>


In [4]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

import numpy as np


@dataclass
class PolicyIterationResult:
    policy: np.ndarray                  # shape: (n_states,), deterministic action per state
    value: np.ndarray                   # shape: (n_states,)
    policy_stable: bool
    n_policy_iterations: int
    eval_iterations_per_round: list[int]


def _q_values(
    P: np.ndarray,
    R: np.ndarray,
    V: np.ndarray,
    gamma: float,
) -> np.ndarray:
    """
    Compute Q(s,a) for all states/actions.

    P shape: (S, A, S)
    R shape: (S, A) or (S, A, S)
    V shape: (S,)
    """
    if R.ndim == 2:
        # Q(s,a) = R(s,a) + gamma * sum_{s'} P(s'|s,a) V(s')
        return R + gamma * np.einsum("sas,s->sa", P, V)
    if R.ndim == 3:
        # Q(s,a) = sum_{s'} P(s'|s,a) * (R(s,a,s') + gamma * V(s'))
        return np.einsum("sas,sas->sa", P, R + gamma * V[None, None, :])

    raise ValueError("R must have shape (S, A) or (S, A, S)")


def iterative_policy_evaluation(
    P: np.ndarray,
    R: np.ndarray,
    policy: np.ndarray,
    gamma: float = 0.9,
    theta: float = 1e-8,
    max_iterations: int = 10_000,
    V0: Optional[np.ndarray] = None,
) -> tuple[np.ndarray, int]:
    """
    Iterative policy evaluation for deterministic policy.

    Returns:
        V, n_iterations
    """
    n_states, n_actions, n_states_2 = P.shape
    if n_states != n_states_2:
        raise ValueError("P must have shape (S, A, S)")
    if policy.shape != (n_states,):
        raise ValueError("policy must have shape (S,)")
    if not np.issubdtype(policy.dtype, np.integer):
        raise ValueError("policy must contain integer action indices")
    if np.any(policy < 0) or np.any(policy >= n_actions):
        raise ValueError("policy contains invalid action index")

    V = np.zeros(n_states, dtype=float) if V0 is None else V0.astype(float, copy=True)

    for k in range(1, max_iterations + 1):
        delta = 0.0
        V_new = V.copy()

        for s in range(n_states):
            a = policy[s]
            if R.ndim == 2:
                v_sa = R[s, a] + gamma * np.dot(P[s, a], V)
            elif R.ndim == 3:
                v_sa = np.dot(P[s, a], R[s, a] + gamma * V)
            else:
                raise ValueError("R must have shape (S, A) or (S, A, S)")

            V_new[s] = v_sa
            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new
        if delta < theta:
            return V, k

    return V, max_iterations


def policy_iteration(
    P: np.ndarray,
    R: np.ndarray,
    gamma: float = 0.9,
    theta: float = 1e-8,
    max_eval_iterations: int = 10_000,
    max_policy_iterations: int = 1_000,
    initial_policy: Optional[np.ndarray] = None,
) -> PolicyIterationResult:
    """
    Policy iteration using iterative policy evaluation.

    Args:
        P: Transition probabilities, shape (S, A, S)
        R: Rewards, shape (S, A) or (S, A, S)
        gamma: Discount factor
        theta: Convergence threshold for policy evaluation
        max_eval_iterations: Max sweeps during each policy evaluation
        max_policy_iterations: Max policy improvement rounds
        initial_policy: Optional deterministic initial policy, shape (S,)

    Returns:
        PolicyIterationResult
    """
    n_states, n_actions, _ = P.shape

    if initial_policy is None:
        policy = np.zeros(n_states, dtype=int)
    else:
        policy = initial_policy.astype(int, copy=True)
        if policy.shape != (n_states,):
            raise ValueError("initial_policy must have shape (S,)")

    V = np.zeros(n_states, dtype=float)
    eval_iterations_per_round: list[int] = []
    stable = False

    for i in range(1, max_policy_iterations + 1):
        # 1) Policy Evaluation (iterative)
        V, n_eval = iterative_policy_evaluation(
            P=P,
            R=R,
            policy=policy,
            gamma=gamma,
            theta=theta,
            max_iterations=max_eval_iterations,
            V0=V,
        )
        eval_iterations_per_round.append(n_eval)

        # 2) Policy Improvement (greedy wrt current V)
        q = _q_values(P, R, V, gamma)
        new_policy = np.argmax(q, axis=1)

        if np.array_equal(new_policy, policy):
            stable = True
            return PolicyIterationResult(
                policy=policy,
                value=V,
                policy_stable=stable,
                n_policy_iterations=i,
                eval_iterations_per_round=eval_iterations_per_round,
            )

        policy = new_policy

    return PolicyIterationResult(
        policy=policy,
        value=V,
        policy_stable=stable,
        n_policy_iterations=max_policy_iterations,
        eval_iterations_per_round=eval_iterations_per_round,
    )

In [6]:
# Build (S, A, S) and (S, A) arrays from the dictionary MDP, then call policy_iteration

action_list = sorted({a for s in S if s not in terminal_states_mdp for a in A[s]})
state_to_idx = {s: i for i, s in enumerate(S)}
action_to_idx = {a: i for i, a in enumerate(action_list)}

n_states = len(S)
n_actions = len(action_list)

P = np.zeros((n_states, n_actions, n_states), dtype=float)
R = np.full((n_states, n_actions), -100.0, dtype=float)  # discourage invalid actions

for s in S:
    s_idx = state_to_idx[s]

    if s in terminal_states_mdp:
        # Terminal state: all actions are equivalent self-loop with zero reward
        for a_idx in range(n_actions):
            P[s_idx, a_idx, s_idx] = 1.0
            R[s_idx, a_idx] = 0.0
        continue

    valid_actions = set(actions(s))

    for a in action_list:
        a_idx = action_to_idx[a]

        if a not in valid_actions:
            # Invalid action in this state -> self-loop with poor reward
            P[s_idx, a_idx, s_idx] = 1.0
            continue

        R[s_idx, a_idx] = reward_sa(s, a)
        for prob, s_next in transitions_sa(s, a):
            P[s_idx, a_idx, state_to_idx[s_next]] += prob

result = policy_iteration(P=P, R=R, gamma=0.9, theta=1e-12)

print("Policy iteration converged:", result.policy_stable)
print("Policy improvement rounds:", result.n_policy_iterations)
print("Evaluation iterations per round:", result.eval_iterations_per_round)

print("\nOptimal deterministic policy:")
for s in S:
    if s in terminal_states_mdp:
        print(f"  {s:>5} -> (terminal)")
        continue
    a_idx = result.policy[state_to_idx[s]]
    print(f"  {s:>5} -> {action_list[a_idx]}")

print("\nOptimal state values:")
for s in S:
    print(f"  {s:>5}: {result.value[state_to_idx[s]]: .6f}")

Policy iteration converged: True
Policy improvement rounds: 2
Evaluation iterations per round: [308, 264]

Optimal deterministic policy:
     C1 -> facebook
     C2 -> sleep
     C3 -> study
     FB -> quit
  Sleep -> (terminal)

Optimal state values:
     C1: -5.263158
     C2:  0.000000
     C3:  10.000000
     FB: -4.736842
  Sleep:  0.000000
